In [3]:
import tkinter as tk
from tkinter import ttk #Combobox
from tkinter import messagebox
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from datetime import datetime, timedelta, timezone
#---------------------------------------------------------------------------------
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write

def record():
    # ----------------------------
    # Settings
    # ----------------------------
    duration = 10  # seconds to record
    fs = 16000    # sampling frequency (Hz)
    filename = "recorded_audio.wav"  # output file

    # ----------------------------
    # Record audio
    # ----------------------------
    print("Recording...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()  # Wait until recording is finished
    print("Recording finished!")

    # ----------------------------
    # Save as WAV file
    # ----------------------------
    # Convert to int16 for WAV format
    write(filename, fs, np.int16(recording * 32767))
    print(f"Audio saved as {filename}")
    speech_to_text()

#---------------------------------------------------------------------------------

import wave
import json
from vosk import Model, KaldiRecognizer

def speech_to_text():
    

    # Load the Vosk model
    model_path = r"C:\Users\graem\Downloads\vosk-model-small-en-us-0.15\vosk-model-small-en-us-0.15"
    model = Model(model_path)

    # Open recorded audio
    wf = wave.open("recorded.wav", "rb")

    # Initialize recognizer
    rec = KaldiRecognizer(model, wf.getframerate())

    transcription = ""

    # Transcribe
    while True:
        data = wf.readframes(4000)

        if len(data) == 0:
            break

        if rec.AcceptWaveform(data):
            result = json.loads(rec.Result())
            transcription += result.get("text", "") + " "

    # Final result
    result = json.loads(rec.FinalResult())
    transcription += result.get("text", "")
    
    text_notes.insert(tk.END, transcription)

    return transcription


# Run transcription
#---------------------------------------------------------------------------------
import json
import threading
import wave
import queue

# Declare model path
model_path = r"C:\Users\graem\Downloads\vosk-model-small-en-us-0.15\vosk-model-small-en-us-0.15"
model = Model(model_path)

# Global control flag
global listening
listening = False

global audio_frames
audio_frames = []
q = queue.Queue()

#---------------------------------------------------------------------------------
# Function for speech
def audio_thread():
    global listening, audio_frames

    rec = KaldiRecognizer(model, 16000)
    audio_frames = []

    def callback(indata, frames, time, status):
        if listening:
            data = bytes(indata)
            audio_frames.append(data)

            if rec.AcceptWaveform(data):
                result = json.loads(rec.Result())
                q.put(result.get("text", ""))

    with sd.RawInputStream(samplerate=16000,
                           blocksize=8000,
                           dtype='int16',
                           channels=1,
                           callback=callback):
        while listening:
            sd.sleep(100)   # IMPORTANT: prevents CPU lock
#---------------------------------------------------------------------------------
def update_textbox():
    """Safely update UI from queue"""
    try:
        while True:
            text = q.get_nowait()
            text_notes.insert(tk.END, text + "\n")
    except queue.Empty:
        pass
    root.after(100, update_textbox)

#---------------------------------------------------------------------------------
def save_wav(filename="recorded.wav"):
    wf = wave.open(filename, 'wb')
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(16000)
    wf.writeframes(b''.join(audio_frames))
    wf.close()

#---------------------------------------------------------------------------------
# Start Recording
def start_listening():
    global listening
    listening = True
    print("Listening...")
    threading.Thread(target=audio_thread, daemon=True).start()

#---------------------------------------------------------------------------------
# Stop Recording
def stop_listening():
    global listening
    listening = False

    final = json.loads(KaldiRecognizer(model, 16000).FinalResult())
    q.put("FINAL: " + final.get("text", ""))
    
    save_wav()
    print("Saved to recorded.wav")

    speech_to_text()


#---------------------------------------------------------------------------------
# Create PDF from data on the form
def generate_pdf():
    name = entry_name.get()
    age = entry_age.get()
    gender = combo_gender.get()
    diagnosis = entry_diagnosis.get()
    bp = entry_bp.get()
    avgheartrate = entry_avgheartrate.get()
    avgspo2 = entry_avgspo2.get()
    avgsleephours = entry_avgsleephours.get()
    avgcalorieintake = entry_avgcalorieintake.get()
    usualdietcompliance = entry_usualdietcompliance.get()
    patientid = patient_id.get()
    dt = datetime.get()
    
    notes = text_notes.get("1.0", tk.END).strip()

    #if not name or not email:
        #messagebox.showerror("Error", "Name and Email are required!")
        #return

    if not name:
        messagebox.showerror("Error", "Name is required!")
        return

    pdf = SimpleDocTemplate("form_output.pdf")
    styles = getSampleStyleSheet()

    content = []

    # ✅ 1. Add Logo (make sure logo.png exists in same folder)
    try:
        logo = Image("logo.png", width=120, height=60)
        content.append(logo)
        content.append(Spacer(1, 10))
    except:
        pass  # skip if logo not found

    # Title
    content.append(Paragraph("<b>User Information Report</b>", styles["Title"]))
    content.append(Spacer(1, 20))

    # ✅ 2. Table Layout
    data = [
        ["Field", "Value"],
        ["Patient ID", patientid],
        ["Date / Time", dt],
        ["Name", name],
        ["Age", age],
        ["Gender", gender],
        ["Diagnosis", diagnosis],
        ["Blood Pressure", bp],
        ["Avg. Heart Rate", avgheartrate],
        ["Avg. SPO2", avgspo2],
        ["Avg. Sleep Hours", avgsleephours],
        ["Avg. Calorie Intake", avgcalorieintake],
        ["Usual Diet Compliance", usualdietcompliance]
    ]

    table = Table(data, colWidths=[100, 250])

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.grey),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),

        ("GRID", (0, 0), (-1, -1), 1, colors.black),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),

        ("ALIGN", (0, 0), (-1, -1), "LEFT"),
        ("PADDING", (0, 0), (-1, -1), 8)
    ]))

    content.append(table)
    content.append(Spacer(1, 20))

    # Notes Section
    content.append(Paragraph("<b>Notes:</b>", styles["Normal"]))
    content.append(Spacer(1, 10))
    content.append(Paragraph(notes, styles["Normal"]))

    # Build PDF
    pdf.build(content)

    messagebox.showinfo("Success", "PDF Generated Successfully!")
#---------------------------------------------------------------------------------
def close_form():
        form.destroy()
#---------------------------------------------------------------------------------
# Tkinter UI
root = tk.Tk()
form = tk.Toplevel(root)
form.title("Medical Diagnosis Form")
form.geometry("800x800")
form.configure(bg="#f4f6f7")

# --- Make a pop-up modal ---
form.transient(root)
#form.grab_set()
form.focus_force()
form.lift()

# --- Title ---
tk.Label(form, text="Medical Diagnosis Form",
            font=("Arial", 16, "bold"),
            bg="#f4f6f7").pack(pady=15)

# --- Container ---
container = tk.Frame(form, bg="white", bd=1, relief="solid")
container.pack(padx=20, pady=10, fill="both", expand=True)

container.columnconfigure(0, weight=1, minsize=150)
container.columnconfigure(1, weight=2, minsize=300)

# --- Labels ---
labels = ["Patient ID", "Date / Time", "Patient Name *", "Age", "Gender", "Diagnosis *", "Blood Pressure",  "Nursing Notes", 
          "Avg. Heart Rate", "Avg. SPO2","Avg. Sleep Hours", "Avg. Calorie Intake", "Usual Diet Compliance"]

for i, text in enumerate(labels):
    tk.Label(container, text=text, anchor="e", bg="white", wraplength=150, justify="right")\
        .grid(row=i, column=0, padx=15, pady=10, sticky="e")

# --- Inputs ---

patient_id = tk.Entry(container)
patient_id.insert(0,"0011773333")
patient_id.grid(row=0, column=1, padx=15, pady=10, sticky="ew")
patient_id.config(state="readonly")  # Makes it read-only

now = datetime.now()

datetime = tk.Entry(container)
datetime.insert(1,now.strftime("%d/%m/%Y %H:%M:%S"))
datetime.grid(row=1, column=1, padx=15, pady=10, sticky="ew")
datetime.config(state="readonly")  # Makes it read-only

entry_name = tk.Entry(container)
entry_name.insert(2,"John Smith")
entry_name.grid(row=2, column=1, padx=15, pady=10, sticky="ew")
name = entry_name.get()

entry_age = tk.Entry(container)
entry_age.insert(3,"35")
entry_age.grid(row=3, column=1, padx=15, pady=10, sticky="ew")
age = entry_age.get()

combo_gender = ttk.Combobox(container, values=["Male", "Female", "Other"])
combo_gender.grid(row=4, column=1, padx=15, pady=10, sticky="ew")

entry_diagnosis = tk.Entry(container)
entry_diagnosis.grid(row=5, column=1, padx=15, pady=10, sticky="ew")

entry_bp = tk.Entry(container)
entry_bp.grid(row=6, column=1, padx=15, pady=10, sticky="ew")

text_notes = tk.Text(container, height=5)
text_notes.grid(row=7, column=1, padx=15, pady=10, sticky="ew")

entry_avgheartrate = tk.Entry(container)
entry_avgheartrate.insert(8,"120 bpm")
entry_avgheartrate.grid(row=8, column=1, padx=15, pady=10, sticky="ew")
entry_avgheartrate.config(state="readonly")  # Makes it read-only

entry_avgspo2 = tk.Entry(container)
entry_avgspo2.insert(9,"Test")
entry_avgspo2.grid(row=9, column=1, padx=15, pady=10, sticky="ew")
entry_avgspo2.config(state="readonly")  # Makes it read-only

entry_avgsleephours = tk.Entry(container)
entry_avgsleephours.insert(10,"Test Sleep Hours")
entry_avgsleephours.grid(row=10, column=1, padx=15, pady=10, sticky="ew")
entry_avgsleephours.config(state="readonly")  # Makes it read-only

entry_avgcalorieintake = tk.Entry(container)
entry_avgcalorieintake.insert(11,"Test Calorie Intake")
entry_avgcalorieintake.grid(row=11, column=1, padx=15, pady=10, sticky="ew")
entry_avgcalorieintake.config(state="readonly")  # Makes it read-only

entry_usualdietcompliance = tk.Entry(container)
entry_usualdietcompliance.insert(12,"Test Usual Diet Compliance")
entry_usualdietcompliance.grid(row=12, column=1, padx=15, pady=10, sticky="ew")
entry_usualdietcompliance.config(state="readonly")  # Makes it read-only

    
# --- Buttons ---
btn_frame = tk.Frame(form, bg="#f4f6f7")
btn_frame.pack(pady=15)


#tk.Button(btn_frame, text="Submit", command=submit_form)\
    #.grid(row=0, column=0, padx=15)

tk.Button(btn_frame, text="Close", command=close_form)\
    .grid(row=0, column=1, padx=15)
    
tk.Button(btn_frame, text="Record", command=record)\
    .grid(row=1, column=0, padx=15)
    
tk.Button(btn_frame, text="Generate PDF", command=generate_pdf)\
    .grid(row=1, column=0, padx=15)

tk.Button(btn_frame, text="Start Recording", command=start_listening)\
    .grid(row=1, column=1, padx=15)

tk.Button(btn_frame, text="Stop Recording", command=stop_listening)\
    .grid(row=1, column=2, padx=15)

root.mainloop()
